# **Semana 14: Plan de Recuperación ante Desastres (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Simular un plan completo de recuperación ante desastres (DRP - Disaster Recovery Plan) utilizando SQLite. Aprenderemos a realizar backups (copias de seguridad), simular un desastre (pérdida de datos) y restaurar la base de datos a diferentes puntos en el tiempo. Este ejercicio consolida los conceptos de resiliencia, recuperación y buenas prácticas en producción.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Comprender** la importancia de los backups en la estrategia de recuperación.
2. **Implementar** diferentes tipos de backup (completo, incremental, lógico).
3. **Simular** un desastre (pérdida de datos) en una base de datos.
4. **Restaurar** la base de datos desde un backup completo.
5. **Implementar** Point-in-Time Recovery (PITR) utilizando backups incrementales y logs.
6. **Automatizar** un plan de recuperación con scripts en Python.
7. **Reflexionar** sobre métricas clave: RTO (Recovery Time Objective) y RPO (Recovery Point Objective).

## **Configuración Inicial**

Importamos las librerías necesarias y preparamos el entorno.

In [ ]:
# Importaciones
import sqlite3
import pandas as pd
import numpy as np
import os
import shutil
import time
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import hashlib

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

# Crear directorios para backups
os.makedirs("backups", exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Librerías importadas correctamente.")
print(f"Directorios creados: backups/, logs/")

---
## **Parte 1: Conceptos de Recuperación ante Desastres**

### **Métricas clave:**

*   **RTO (Recovery Time Objective)**: Tiempo máximo permitido para restaurar el servicio después de un desastre.
*   **RPO (Recovery Point Objective)**: Pérdida máxima de datos aceptable (en tiempo).

### **Tipos de backup:**

1. **Backup completo (full)**: Copia de toda la base de datos.
2. **Backup incremental**: Solo los cambios desde el último backup (completo o incremental).
3. **Backup diferencial**: Todos los cambios desde el último backup completo.
4. **Backup lógico**: Volcado SQL (con `dump`).
5. **Backup físico**: Copia de los archivos de la base de datos.

### **Estrategias de recuperación:**

*   **Restauración completa**: Desde el último backup completo.
*   **Point-in-Time Recovery (PITR)**: Restaurar a un momento específico usando backups + logs de transacciones.

En este notebook, simularemos estas estrategias con SQLite.

---
## **Parte 2: Crear Base de Datos de Ejemplo**

Crearemos una base de datos SQLite con una tabla de transacciones bancarias para simular un sistema en producción.

In [ ]:
# Crear base de datos principal
DB_FILE = "banco.db"

# Si existe, la eliminamos para empezar limpios
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)

conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Crear tabla de cuentas
cursor.execute('''
    CREATE TABLE cuentas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        titular TEXT NOT NULL,
        saldo REAL NOT NULL,
        fecha_apertura TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        activo BOOLEAN DEFAULT 1
    )
''')

# Crear tabla de transacciones
cursor.execute('''
    CREATE TABLE transacciones (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        cuenta_id INTEGER NOT NULL,
        tipo TEXT CHECK(tipo IN ('DEPOSITO', 'RETIRO', 'TRANSFERENCIA')),
        monto REAL NOT NULL,
        fecha TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        descripcion TEXT,
        FOREIGN KEY (cuenta_id) REFERENCES cuentas(id)
    )
''')

# Insertar datos de ejemplo
cuentas_ejemplo = [
    ('Ana García', 5000),
    ('Carlos López', 3500),
    ('María Rodríguez', 8200),
    ('Juan Martínez', 1200),
    ('Laura Sánchez', 4300)
]

for titular, saldo in cuentas_ejemplo:
    cursor.execute('INSERT INTO cuentas (titular, saldo) VALUES (?, ?)', (titular, saldo))

conn.commit()

# Verificar datos
df_cuentas = pd.read_sql_query("SELECT * FROM cuentas", conn)
print("Cuentas creadas:")
display(df_cuentas)

conn.close()
print(f"\n✅ Base de datos '{DB_FILE}' creada.")

---
## **Parte 3: Simulación de Actividad (Generación de Datos)**

Simularemos actividad en la base de datos durante varios días para tener puntos de recuperación.

In [ ]:
def simular_actividad(dias=5, transacciones_por_dia=20):
    """Simula actividad durante varios días"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # Obtener IDs de cuentas activas
    cursor.execute("SELECT id FROM cuentas WHERE activo = 1")
    cuentas = [row[0] for row in cursor.fetchall()]
    
    fecha_base = datetime.now() - timedelta(days=dias)
    
    for dia in range(dias):
        fecha_dia = fecha_base + timedelta(days=dia)
        
        for _ in range(transacciones_por_dia):
            cuenta_id = np.random.choice(cuentas)
            tipo = np.random.choice(['DEPOSITO', 'RETIRO', 'TRANSFERENCIA'], p=[0.4, 0.4, 0.2])
            monto = round(np.random.uniform(10, 500), 2)
            
            if tipo == 'RETIRO':
                monto = -monto
            elif tipo == 'TRANSFERENCIA':
                # Transferencia entre cuentas (simplificada)
                cuenta_destino = np.random.choice([c for c in cuentas if c != cuenta_id])
                monto = -monto
                # No actualizamos saldo aquí para simplificar
            
            # Insertar transacción con fecha específica
            cursor.execute('''
                INSERT INTO transacciones (cuenta_id, tipo, monto, fecha, descripcion)
                VALUES (?, ?, ?, ?, ?)
            ''', (cuenta_id, tipo, abs(monto) if tipo == 'DEPOSITO' else -abs(monto), 
                   fecha_dia.strftime('%Y-%m-%d %H:%M:%S'), f"Transacción día {dia+1}"))
            
            # Actualizar saldo (simplificado)
            cursor.execute('UPDATE cuentas SET saldo = saldo + ? WHERE id = ?', 
                          (monto if tipo != 'TRANSFERENCIA' else -abs(monto), cuenta_id))
        
        conn.commit()
        print(f"  Día {dia+1}: {transacciones_por_dia} transacciones insertadas")
    
    conn.close()
    return dias * transacciones_por_dia

print("Simulando actividad en la base de datos...")
total_trans = simular_actividad(dias=5, transacciones_por_dia=15)
print(f"✅ Actividad simulada: {total_trans} transacciones insertadas.")

---
## **Parte 4: Estrategias de Backup**

### **4.1. Backup completo (Full backup)**

In [ ]:
def backup_completo(nombre=None):
    """Realiza un backup completo de la base de datos"""
    if nombre is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        nombre = f"backup_completo_{timestamp}.db"
    
    backup_path = os.path.join("backups", nombre)
    
    # Copiar archivo (backup físico)
    shutil.copy2(DB_FILE, backup_path)
    
    # Registrar metadata
    metadata = {
        'fecha': datetime.now().isoformat(),
        'tipo': 'completo',
        'archivo': nombre,
        'tamaño_bytes': os.path.getsize(backup_path),
        'md5': hashlib.md5(open(backup_path, 'rb').read()).hexdigest()
    }
    
    with open(os.path.join("backups", f"{nombre}.meta"), 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"✅ Backup completo creado: {backup_path}")
    print(f"   Tamaño: {metadata['tamaño_bytes']/1024:.2f} KB")
    print(f"   MD5: {metadata['md5']}")
    
    return backup_path, metadata

# Realizar backup completo inicial
backup1_path, meta1 = backup_completo("backup_inicial.db")

# Mostrar metadata
print("\nMetadata del backup:")
print(json.dumps(meta1, indent=2))

### **4.2. Backup lógico (SQL dump)**

SQLite permite hacer un volcado SQL (backup lógico) que es portable y legible.

In [ ]:
def backup_logico():
    """Realiza un backup lógico (dump SQL)"""
    conn = sqlite3.connect(DB_FILE)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    dump_file = os.path.join("backups", f"dump_{timestamp}.sql")
    
    with open(dump_file, 'w') as f:
        for line in conn.iterdump():
            f.write('%s\n' % line)
    
    conn.close()
    print(f"✅ Backup lógico creado: {dump_file}")
    print(f"   Tamaño: {os.path.getsize(dump_file)/1024:.2f} KB")
    return dump_file

dump_file = backup_logico()

### **4.3. Backup incremental (simulado)**

SQLite no tiene soporte nativo para backups incrementales. Simularemos esto guardando un registro de cambios (como un log de transacciones) y realizando backups completos periódicos.

In [ ]:
def registrar_cambio_log(tabla, operacion, datos):
    """Registra un cambio en un archivo de log (simula WAL/binlog)"""
    log_entry = {
        'timestamp': datetime.now().isoformat(),
        'tabla': tabla,
        'operacion': operacion,
        'datos': datos
    }
    
    log_file = os.path.join("logs", "cambios.log")
    with open(log_file, 'a') as f:
        f.write(json.dumps(log_entry) + '\n')
    
    return log_entry

# Ejemplo: registrar cambios futuros
registrar_cambio_log('cuentas', 'INSERT', {'id': 6, 'titular': 'Pedro Gómez', 'saldo': 2500})
registrar_cambio_log('cuentas', 'UPDATE', {'id': 1, 'saldo_anterior': 5000, 'saldo_nuevo': 5200})
print("✅ Cambios registrados en logs/cambios.log")

---
## **Parte 5: Simulación de Desastre**

Simularemos un desastre: la base de datos principal se corrompe o elimina.

In [ ]:
print("💥 SIMULANDO DESASTRE...")
time.sleep(1)

# Eliminar la base de datos principal
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)
    print(f"❌ Base de datos '{DB_FILE}' eliminada (simulación de desastre).")
else:
    print(f"⚠️ La base de datos '{DB_FILE}' ya no existe.")

# Verificar que ya no existe
print(f"\n¿Existe la BD? {os.path.exists(DB_FILE)}")

---
## **Parte 6: Plan de Recuperación**

### **6.1. Restauración desde backup completo**

In [ ]:
def restaurar_backup(backup_path):
    """Restaura la base de datos desde un backup completo"""
    start_time = time.time()
    
    # Verificar integridad del backup (simulado con MD5)
    if os.path.exists(backup_path + ".meta"):
        with open(backup_path + ".meta", 'r') as f:
            metadata = json.load(f)
        
        md5_actual = hashlib.md5(open(backup_path, 'rb').read()).hexdigest()
        if md5_actual != metadata['md5']:
            print("❌ Error: El backup está corrupto (MD5 no coincide)")
            return False
        print("✅ Integridad del backup verificada.")
    
    # Restaurar copiando el archivo
    shutil.copy2(backup_path, DB_FILE)
    
    restore_time = time.time() - start_time
    print(f"✅ Base de datos restaurada desde: {backup_path}")
    print(f"   Tiempo de restauración: {restore_time:.3f} segundos")
    
    # Verificar la restauración
    try:
        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM cuentas")
        count = cursor.fetchone()[0]
        conn.close()
        print(f"   Tabla 'cuentas' contiene {count} registros.")
    except Exception as e:
        print(f"   ❌ Error al verificar: {e}")
    
    return True

# Restaurar desde el backup inicial
restaurar_backup(backup1_path)

### **6.2. Verificar estado después de restauración**

In [ ]:
# Conectar a la base restaurada
conn = sqlite3.connect(DB_FILE)

# Mostrar cuentas
df_cuentas_restauradas = pd.read_sql_query("SELECT * FROM cuentas", conn)
print("Cuentas después de restauración:")
display(df_cuentas_restauradas)

# Mostrar transacciones (últimas 5)
df_trans = pd.read_sql_query("""
    SELECT t.*, c.titular 
    FROM transacciones t
    JOIN cuentas c ON t.cuenta_id = c.id
    ORDER BY t.fecha DESC
    LIMIT 5
""", conn)
print("\nÚltimas 5 transacciones:")
display(df_trans)

conn.close()

### **6.3. Pérdida de datos (RPO)**

Observamos que los datos generados DESPUÉS del backup se perdieron. Calculemos el RPO.

In [ ]:
# Obtener fecha del backup
backup_fecha = datetime.fromisoformat(meta1['fecha'])
print(f"📅 Fecha del backup: {backup_fecha}")

# Conectar a la base restaurada (solo datos hasta backup)
conn = sqlite3.connect(DB_FILE)

# Obtener fecha de la última transacción en la BD restaurada
df_max_fecha = pd.read_sql_query("SELECT MAX(fecha) as ultima_fecha FROM transacciones", conn)
ultima_fecha_bd = pd.to_datetime(df_max_fecha['ultima_fecha'][0])
print(f"📅 Última transacción en BD restaurada: {ultima_fecha_bd}")

# Calcular RPO (datos perdidos en tiempo)
rpo = (datetime.now() - ultima_fecha_bd).total_seconds() / 3600  # en horas
print(f"\n⏱️ RPO (Recovery Point Objective): {rpo:.2f} horas de datos perdidos.")
print("   Esto significa que perdimos todas las transacciones posteriores al backup.")

conn.close()

---
## **Parte 7: Point-in-Time Recovery (PITR) Simulado**

Para mejorar el RPO, necesitamos poder recuperar a un punto específico usando backups incrementales + logs.

### **7.1. Generar logs de cambios**

Simularemos que después del backup, seguimos registrando cambios en un archivo de log.

In [ ]:
def generar_logs_posteriores():
    """Genera logs de cambios después del backup"""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # Obtener estado actual de cuentas
    cursor.execute("SELECT id, titular, saldo FROM cuentas")
    cuentas = cursor.fetchall()
    
    # Simular cambios posteriores
    cambios = []
    
    # Cambio 1: Actualizar saldo de Ana (cuenta 1)
    cambios.append(('cuentas', 'UPDATE', {'id': 1, 'nuevo_saldo': 5500}))
    
    # Cambio 2: Nueva cuenta
    cambios.append(('cuentas', 'INSERT', {'id': 6, 'titular': 'Pedro Gómez', 'saldo': 2500}))
    
    # Cambio 3: Transacción
    cambios.append(('transacciones', 'INSERT', {
        'cuenta_id': 2, 
        'tipo': 'RETIRO', 
        'monto': -200, 
        'descripcion': 'Retiro en cajero'
    }))
    
    for cambio in cambios:
        registrar_cambio_log(cambio[0], cambio[1], cambio[2])
    
    conn.close()
    print(f"✅ {len(cambios)} cambios registrados en log.")

# Generar logs posteriores
generar_logs_posteriores()

### **7.2. Leer logs de cambios**

In [ ]:
def leer_logs(archivo_log="logs/cambios.log"):
    """Lee y retorna las entradas del log"""
    if not os.path.exists(archivo_log):
        return []
    
    with open(archivo_log, 'r') as f:
        lineas = f.readlines()
    
    logs = []
    for linea in lineas:
        try:
            logs.append(json.loads(linea.strip()))
        except:
            continue
    
    return logs

logs = leer_logs()
print(f"Se encontraron {len(logs)} entradas en el log.")
for i, log in enumerate(logs[-5:]):  # Últimas 5
    print(f"{i+1}. {log}")

### **7.3. Simular PITR aplicando logs**

Para recuperar a un punto específico, necesitaríamos:
1. Restaurar desde el último backup completo.
2. Aplicar los logs hasta el momento deseado.

Aquí simulamos la aplicación de logs a una base de datos recién restaurada.

In [ ]:
def aplicar_logs_hasta_fecha(backup_path, fecha_objetivo, logs):
    """Aplica logs hasta una fecha objetivo"""
    # 1. Restaurar backup
    restaurar_backup(backup_path)
    
    # 2. Conectar a BD restaurada
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    
    # 3. Filtrar logs hasta fecha objetivo
    logs_aplicar = [log for log in logs 
                    if datetime.fromisoformat(log['timestamp']) <= fecha_objetivo]
    
    print(f"\nAplicando {len(logs_aplicar)} logs hasta {fecha_objetivo}...")
    
    # 4. Aplicar logs en orden
    for log in logs_aplicar:
        tabla = log['tabla']
        op = log['operacion']
        datos = log['datos']
        
        if tabla == 'cuentas':
            if op == 'INSERT':
                cursor.execute('''
                    INSERT OR REPLACE INTO cuentas (id, titular, saldo)
                    VALUES (?, ?, ?)
                ''', (datos['id'], datos['titular'], datos['saldo']))
            elif op == 'UPDATE':
                cursor.execute('''
                    UPDATE cuentas SET saldo = ? WHERE id = ?
                ''', (datos.get('nuevo_saldo', datos.get('saldo_nuevo')), datos['id']))
        elif tabla == 'transacciones' and op == 'INSERT':
            # Insertar transacción
            cursor.execute('''
                INSERT INTO transacciones (cuenta_id, tipo, monto, descripcion, fecha)
                VALUES (?, ?, ?, ?, ?)
            ''', (datos['cuenta_id'], datos['tipo'], datos['monto'], 
                   datos.get('descripcion', ''), log['timestamp']))
    
    conn.commit()
    conn.close()
    print(f"✅ Logs aplicados hasta {fecha_objetivo}")

# Definir fecha objetivo (ahora menos 1 hora, simulando querer recuperar a ese punto)
fecha_objetivo = datetime.now() - timedelta(hours=1)
aplicar_logs_hasta_fecha(backup1_path, fecha_objetivo, logs)

### **7.4. Verificar PITR**

In [ ]:
conn = sqlite3.connect(DB_FILE)

# Verificar cuentas después de PITR
df_cuentas_pitr = pd.read_sql_query("SELECT * FROM cuentas ORDER BY id", conn)
print("Cuentas después de PITR:")
display(df_cuentas_pitr)

# Verificar transacciones recientes
df_trans_pitr = pd.read_sql_query("""
    SELECT t.*, c.titular 
    FROM transacciones t
    JOIN cuentas c ON t.cuenta_id = c.id
    ORDER BY t.fecha DESC
    LIMIT 10
""", conn)
print("\nTransacciones después de PITR:")
display(df_trans_pitr)

conn.close()

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica y extiende los conceptos de recuperación.

### **Ejercicio 1: Backup programado automático**

Crea una función `backup_programado` que:
1. Realice un backup completo cada cierto intervalo (simulado con `time.sleep`).
2. Mantenga solo los últimos N backups (rotación).
3. Registre en un archivo de log las operaciones de backup.

*Pista: Puedes usar `os.listdir` y `os.path.getmtime` para gestionar la rotación.*

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def backup_programado(intervalo_segundos=10, max_backups=3):
    """Simula backups programados"""
    pass

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Restauración con verificación de integridad**

Mejora la función `restaurar_backup` para:
1. Verificar que el backup no esté corrupto (usando MD5).
2. Antes de restaurar, hacer un backup automático del estado actual (por si acaso).
3. Después de restaurar, ejecutar consultas de verificación (`PRAGMA integrity_check` en SQLite).

In [ ]:
# --- INICIO DE TU CÓDIGO ---

def restaurar_backup_mejorado(backup_path):
    # 1. Verificar integridad del backup
    # ...
    # 2. Backup automático de seguridad
    # ...
    # 3. Restaurar
    # ...
    # 4. Verificar con PRAGMA
    # ...
    pass

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Simular pérdida de datos y recuperación con logs**

1. Realiza un backup completo.
2. Simula 10 nuevas transacciones.
3. Registra cada transacción en el archivo de logs.
4. Elimina la base de datos (simula desastre).
5. Restaura desde el backup y aplica los logs para recuperar las 10 transacciones.

Mide el RTO y RPO de este proceso.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# 1. Backup completo
# ...

# 2. Simular transacciones y registrar logs
# ...

# 3. Desastre
# ...

# 4. Recuperación
# ...

# 5. Medir RTO y RPO
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 4: Reflexión final del curso**

En una celda Markdown, responde:

1. **¿Qué estrategias de backup implementarías en una base de datos de producción?**
   *   Frecuencia de backups completos vs incrementales.
   *   Políticas de retención.
   *   Backups en múltiples ubicaciones (on-premise y cloud).

2. **¿Cómo afectan RTO y RPO al diseño de tu estrategia de recuperación?**

3. **¿Qué herramientas de la nube (AWS RDS, Azure SQL, etc.) ofrecen para automatizar backups y PITR?**

4. **¿Cuál ha sido el aprendizaje más importante del curso para ti?**

---
## **Conclusiones del Curso**

Hemos completado un viaje de 14 semanas a través del mundo de las bases de datos para inteligencia artificial:

| Semana | Tema |
|--------|------|
| 1-2 | Modelado relacional y normalización |
| 3-4 | SQL avanzado y consultas |
| 5-6 | Índices y optimización |
| 7 | Limpieza de datos |
| 8 | Captura de datos (scraping, APIs, CDC) |
| 9 | Pipelines ETL/ELT, batch y streaming |
| 10 | Big Data: Spark y Databricks |
| 11 | NoSQL: MongoDB, Redis, Neo4j |
| 12 | Bases vectoriales y RAG |
| 13 | Docker y alta disponibilidad |
| 14 | Gestión de errores y arquitecturas híbridas |

**Hoy, en la última sesión, hemos aprendido a protegernos contra lo peor: la pérdida de datos.**

*   Backups completos, incrementales y lógicos.
*   Point-in-Time Recovery.
*   Métricas RTO y RPO.
*   Plan de recuperación ante desastres.

**Recuerda:** En producción, no es cuestión de *si* ocurrirá un desastre, sino de *cuándo*. Estar preparado marca la diferencia entre una interrupción menor y una catástrofe empresarial.

¡Felicitaciones por completar el curso!

In [ ]:
# Limpieza opcional (comentar si quieres conservar los archivos)
# shutil.rmtree("backups")
# shutil.rmtree("logs")
# if os.path.exists(DB_FILE):
#     os.remove(DB_FILE)
print("\n📌 Nota: Los archivos de backup y logs se han conservado para revisión.")